# Data Preparation

In [ ]:
# import packages
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel ('../data/Daten_Siburg.xlsx')
df.head()



In [ ]:
# drop irrelevant column 
df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)
df.drop(labels='Nr', axis=1, inplace=True)
df.head()


In [ ]:
df.info()

In [ ]:
def numCol(df):
    df_numeric = df.select_dtypes(include=[np.number])
    cols = df_numeric.columns.values
    return cols

# apply function
numeric_cols= numCol(df)
print(numeric_cols)

In [ ]:
# drop all column containing only one unique value
# find single unique value column
col_sv=df.columns[df.nunique() == 1]
print('The follwoing column(s) contain only one unique value:',col_sv)

In [ ]:
#count number of missing data per column
df.isnull().sum()

In [ ]:
# conda install -c anaconda seaborn
import seaborn as sns

In [ ]:
# replacing invalid values
df.replace('-', np.nan, inplace=True)
# unkown data must be replaced by nan in python

In [ ]:
# using a heat map to visualize missing data
cols = df.columns # first 30 columns
colours = ['#000098', '#ffff10'] # specify the colours - yellow is missing. blue is not missing.

fig, ax = plt.subplots(figsize=(10,7)) 
sns.heatmap(df[cols].isnull(), cmap=sns.color_palette(colours), ax=ax)
svm.savefig('Missing_Data_Heatmap.png')

In [ ]:
#import MinMaxScaler from sklearn library
from sklearn.preprocessing import MinMaxScaler

# select the properties that should be scaled
cols=['d', 'Fläche', 'rho_l', 'fcm,cyl', 'V_test']

# Create Scaler object
scaler = MinMaxScaler(feature_range=(0.000001, 1))
# fit scaler to data
scaler.fit(df[cols])
#scale chosen properties
df_scaled = pd.DataFrame(scaler.transform(df[cols]), columns=cols)

# inspect distribution
print(df_scaled)

In [ ]:
#Add Stütze to dataframe
Stuetze = df["Stütze"]
df_scaled = df_scaled.join(Stuetze)
print(df_scaled)

In [ ]:
#one-hot enconding Stütze
df_scaled = pd.get_dummies(df_scaled)
print(df_scaled)

In [ ]:
#Add forscher to dataframe
Forscher = df["Forscher"]
df_scaled = df_scaled.join(Forscher)
print(df_scaled)

# Tree Algorithm

**Splitting Data**

Splitting of data into test and train subsets was carried out using the stratify command in order to ensure balanced subsets.

In [ ]:
sns.set(font_scale = 0.5)
sns.set(rc={'figure.figsize':(11.7,8.27)})
plot1=sns.countplot(y='Forscher', data=df_scaled,color='grey', order = df_scaled['Forscher'].value_counts().index)
plot1.figure.savefig("output.png")


In [ ]:
# Deleting Forscher classes with less than 2 members
counts = df_scaled['Forscher'].value_counts()
new_df = df_scaled.loc[df_scaled['Forscher'].isin(counts.index[counts >= 2])]


In [ ]:
sns.set(font_scale = 0.5)
sns.set(rc={'figure.figsize':(11.7,8.27)})
plot2=sns.countplot(y='Forscher', data=new_df,color='grey', order = new_df['Forscher'].value_counts().index)
plot2.figure.savefig("output2.png")


In [ ]:
#Splitting train and test data
from sklearn.model_selection import train_test_split
y = new_df['V_test']
X = new_df.drop(['V_test'],axis=1)

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.3,
                                                    stratify =new_df['Forscher'],
                                                    random_state=88)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
sns.set(font_scale = 0.5)
sns.set(rc={'figure.figsize':(11.7,8.27)})
plot3=sns.countplot(y='Forscher', data=X_train,color='grey', order = X_train['Forscher'].value_counts().index)
plot3.figure.savefig("trainoutput.png")

In [ ]:
sns.set(font_scale = 0.5)
sns.set(rc={'figure.figsize':(11.7,8.27)})
plot4=sns.countplot(y='Forscher', data=X_test,color='grey', order = X_test['Forscher'].value_counts().index)
plot4.figure.savefig("testoutput.png")

In [ ]:
#Remove Forscher from X_train
X_train=X_train.drop(columns=['Forscher'])
print(X_train)

In [ ]:
#Remove Forscher from X_test
X_test=X_test.drop(columns=['Forscher'])
print(X_test)

In [ ]:
matplotlib.rc_file_defaults()

**First Tree Model**


In [ ]:
from sklearn.tree import DecisionTreeRegressor

Tree_1= DecisionTreeRegressor(min_samples_leaf=5,
                             random_state = 88,
                            max_depth=2)

Tree_1 = Tree_1.fit(X_train, y_train)

In [ ]:
# plot trained desision tree
from sklearn.tree import plot_tree

print('Node count =', Tree_1.tree_.node_count)
plt.figure(figsize=(12,12))
plot_tree(Tree_1, 
          feature_names=X_train.columns, 
          class_names=y_train.unique(), 
          filled=True,
          impurity=True,
          rounded=True,
          fontsize=12) 
plt.show()

**Evaluation**

In [ ]:
# make predictions with trained model
y_pred_train = Tree_1.predict(X_train)
y_pred_test = Tree_1.predict(X_test)

In [ ]:
# calculate evaluation metrics
from sklearn.metrics import mean_squared_error

acc_train=mean_squared_error(y_train, y_pred_train)
acc_test=mean_squared_error(y_test, y_pred_test)
print('Acc_train:',round(acc_train,5))
print('Acc_test:',round(acc_test,5))

Tree is too small to capture the patterns in data.

**Second Tree Model**

In [ ]:
Tree_2= DecisionTreeRegressor(min_samples_leaf=5,
                             random_state = 88,
                            max_depth=100)

Tree_2 = Tree_2.fit(X_train, y_train)

In [ ]:
# plot trained desision tree
print('Node count =', Tree_2.tree_.node_count)
plt.figure(figsize=(50,12))
plot_tree(Tree_2, 
          feature_names=X_train.columns, 
          class_names=y_train.unique(), 
          filled=True,
          impurity=True,
          rounded=True,
          fontsize=12) 
plt.show()

**Evaluation Model 2**

In [ ]:
# make predictions with trained model
y_pred_train = Tree_2.predict(X_train)

y_pred_test = Tree_2.predict(X_test)

In [ ]:
# calculate evaluation metrics
from sklearn.metrics import mean_squared_error

acc_train=mean_squared_error(y_train, y_pred_train)
acc_test=mean_squared_error(y_test, y_pred_test)
print('Acc_train:',round(acc_train,5))
print('Acc_test:',round(acc_test,5))

In [ ]:
plt.scatter(y_pred_test  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Tree regressor')

Results got much better, however Test accuracy is lower than train accuracy which means that the model is overfitting. 

**Grid Search with Cross Validation**

In [ ]:
from sklearn.model_selection import GridSearchCV

#Exhaustive search over specified parameter values for an estimator.
grid_values = {'ccp_alpha': np.linspace(0.0, 0.10,20), 
               'min_samples_leaf': [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],
               'min_samples_split': [15,16,17,18,19,20],
               'max_depth': [3,4,5,6,7,8],
               'random_state': [88]} 
            
Tree_3 = DecisionTreeRegressor()

dtc_cv_acc = GridSearchCV(Tree_3, param_grid = grid_values, cv=5, verbose=1) 


#fitting of the regression tree to the data
dtc_cv_acc.fit(X_train, y_train)

In [ ]:
acc = dtc_cv_acc.cv_results_['mean_test_score'] 

ccp = dtc_cv_acc.cv_results_['param_ccp_alpha'].data

pd.DataFrame({'ccp alpha' : ccp, 'Validation Accuracy': acc}).head(20)

In [ ]:
# plot mean accuracy of grid search
plt.figure(figsize=(8, 6))
plt.xlabel('ccp alpha', fontsize=16)
plt.ylabel('mean validation accuracy', fontsize=16)
plt.scatter(ccp, acc, s=2)
plt.plot(ccp, acc, linewidth=3)
plt.grid(True, which='both')
plt.show()

In [ ]:
print('Grid best parameter ccp_alpha (max. accuracy): ', dtc_cv_acc.best_params_['ccp_alpha'])
print('Grid best score (accuracy): ', dtc_cv_acc.best_score_)

In [ ]:
dtc_cv_acc.best_params_

In [ ]:
# plot tree
print('Node count =', dtc_cv_acc.best_estimator_.tree_.node_count)
plt.figure(figsize=(50,12))
plot_tree(dtc_cv_acc.best_estimator_, 
          feature_names=X_train.columns, 
          class_names=y_train.unique(), 
          filled=True,
          impurity=True,
          rounded=True,
          fontsize=12) 
plt.show()

**Evaluation Model 3**

In [ ]:
y_pred_train = dtc_cv_acc.best_estimator_.predict(X_train)
y_pred_test = dtc_cv_acc.best_estimator_.predict(X_test)

In [ ]:
# calculate evaluation metrics
from sklearn.metrics import mean_squared_error

acc_train=mean_squared_error(y_train, y_pred_train)
acc_test=mean_squared_error(y_test, y_pred_test)
print('Acc_train:',round(acc_train,5))
print('Acc_test:',round(acc_test,5))


In [ ]:
plt.scatter(y_pred_test  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Tree regressor')
plt.grid(True,linestyle='-',color='0.75')
print('prediction underestimated meaning on the safe side')
print('MSE %f' %round(acc_test,5))

In [ ]:
#CART feature importance
from matplotlib import pyplot
# get importance
importance = dtc_cv_acc.best_estimator_.feature_importances_
# plot feature importance
pyplot.bar([x for x in range(len(importance))], importance)
pyplot.title('CART Tree regressor feature importance')
bars = ('d', 'Fläche', 'rho_1', 'fm,cyl', 'Stütze_k', 'Stütze_q', 'Stütze_r')
y_pos = np.arange(len(bars))
plt.xticks(y_pos, bars)
pyplot.show()

**Comparison to Euro Code**

In [ ]:
df2 = pd.read_excel ('../data/Data.xlsx')
df2.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# select the properties that should be scaled
cols=['d', 'Fläche', 'rho_l', 'fcm,cyl', 'V_test', 'V_Rd']

# Create Scaler object
scaler = MinMaxScaler(feature_range=(0.000001, 1))
# fit scaler to data
scaler.fit(df2[cols])
#scale chosen properties
df2_scaled = pd.DataFrame(scaler.transform(df2[cols]), columns=cols)

# inspect distribution
print(df2_scaled)


In [ ]:
print(X_test)

In [ ]:
X_test_mod=X_test.drop(columns=['Stütze_k', 'Stütze_q', 'Stütze_r'])
print(X_test_mod)

In [ ]:
#create subset of df2 which has X values similar to x_test
keys = list(X_test_mod.columns.values)
i1 = df2_scaled.set_index(keys).index
i2 = X_test_mod.set_index(keys).index
df_EC=df2_scaled[i1.isin(i2)]
print(df_EC)

In [ ]:
#plt.scatter(x_axis, y_predict, label='prediction')
#plt.scatter(x_axis, y_test, label='actual')
plt.scatter(y_pred_test, y_test, color='b', label='Tree regressor')
plt.scatter(df_EC['V_Rd'], df_EC['V_test'], color='g', label = 'EC')
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'b--', label='V_test = V_predicted')
plt.plot([0,0.8], [0,1], 'g--', label='SF = 1.22')
plt.legend()
plt.show()

In [ ]:
MSE = np.mean(np.square(np.subtract(y_test, y_pred_test)))
print('MSE = ', MSE)
print('scaled SF = ', 1/0.8)

We try to improve the MSE using a random forrest model.

**Random Forest Model**

In [ ]:
#Importing model
from sklearn.ensemble import RandomForestRegressor

**First Random Forest Model**

In [ ]:
# Instantiate model with 1000 decision trees
RF_1 = RandomForestRegressor(n_estimators = 1000, random_state = 88)
# Train the model on training data
RF_1.fit(X_train, y_train);

In [ ]:
# Making predictions with trained model
y_pred_train_rf1 = RF_1.predict(X_train)
y_pred_test_rf1 = RF_1.predict(X_test)
# calculate evaluation metrics
from sklearn.metrics import mean_squared_error

acc_train_rf1=mean_squared_error(y_train, y_pred_train_rf1)
acc_test_rf1=mean_squared_error(y_test, y_pred_test_rf1)
print('Acc_train:',round(acc_train_rf1,5))
print('Acc_test:',round(acc_test_rf1,5))

In [ ]:
plt.scatter(y_pred_test_rf1  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Random Forest')

Model seems to be overfitting

**Second Random Forest with Hyperparameter tuning**

In [ ]:
from sklearn.model_selection import GridSearchCV
#Exhaustive search over specified parameter values for an estimator.
param_grid = {
    'bootstrap': [True],
    'max_depth': [3,4,5,6,7,8],
    'max_features': [2, 3,4,5,6,7],
    'min_samples_leaf': [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],
    'min_samples_split': [15,16,17,18,19,20],
    'n_estimators': [100, 200, 300, 1000]
}
# Create a based model
RF_2 = RandomForestRegressor()
# Instantiate the grid search model
grid_search_RF2 = GridSearchCV(estimator = RF_2, param_grid = param_grid, 
                          cv = 5, n_jobs = -1, verbose = 1)
# Fit the grid search to the data
grid_search_RF2.fit(X_train, y_train)

In [ ]:
grid_search_RF2.best_params_

In [ ]:
y_pred_train_RF2 = grid_search_RF2.best_estimator_.predict(X_train)
y_pred_test_RF2 = grid_search_RF2.best_estimator_.predict(X_test)

In [ ]:
# calculate evaluation metrics
from sklearn.metrics import mean_squared_error

acc_train_RF2=mean_squared_error(y_train, y_pred_train_RF2)
acc_test_RF2=mean_squared_error(y_test, y_pred_test_RF2)
print('Acc_train:',round(acc_train_RF2,5))
print('Acc_test:',round(acc_test_RF2,5))

In [ ]:
plt.scatter(y_pred_test_RF2  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Random Forest')

In [ ]:
#Random Forest feature importance
from matplotlib import pyplot
# get importance
importance2 = grid_search_RF2.best_estimator_.feature_importances_
# plot feature importance
pyplot.bar([x for x in range(len(importance2))], importance2)
pyplot.title('Random Forest feature importance')
bars = ('d', 'Fläche', 'rho_1', 'fm,cyl', 'Stütze_k', 'Stütze_q', 'Stütze_r')
y_pos = np.arange(len(bars))
plt.xticks(y_pos, bars)
pyplot.show()



In [ ]:

print(grid_search_RF2.cv_results_.keys())


In [ ]:
#outputting the results
cvresults = pd.DataFrame(grid_search_RF2.cv_results_)
cvresults

In [ ]:
print(np.mean(cvresults))

In [ ]:
# plot mean accuracy of grid search against max depth
plt.figure(figsize=(8, 6))
plt.xlabel('max depth', fontsize=16)
plt.ylabel('mean test score', fontsize=16)
plt.plot(cvresults['param_max_depth'], cvresults['mean_test_score'], linewidth=3)
plt.grid(True, which='both')
plt.show()


In [ ]:
# plot mean accuracy of grid search against min samples leaf
plt.figure(figsize=(8, 6))
plt.xlabel('min samples leaf', fontsize=16)
plt.ylabel('mean test score', fontsize=16)
plt.plot(cvresults['param_min_samples_leaf'], cvresults['mean_test_score'], linewidth=3)
plt.grid(True, which='both')
plt.show()

In [ ]:
# plot mean accuracy of grid search against min samples split
plt.figure(figsize=(8, 6))
plt.xlabel('min samples split', fontsize=16)
plt.ylabel('mean test score', fontsize=16)
plt.plot(cvresults['param_min_samples_split'], cvresults['mean_test_score'], linewidth=3)
plt.grid(True, which='both')
plt.show()

In [ ]:
# plot mean accuracy of grid search against n_estimators
plt.figure(figsize=(8, 6))
plt.xlabel('number of estimators', fontsize=16)
plt.ylabel('mean test score', fontsize=16)
plt.plot(cvresults['param_n_estimators'], cvresults['mean_test_score'], linewidth=3)
plt.grid(True, which='both')
plt.show()

In [ ]:
# plot a tree in random forest

plt.figure(figsize=(50,12))
plot_tree(grid_search_RF2.best_estimator_[1], 
          feature_names=X_train.columns, 
          class_names=y_train.unique(), 
          filled=True,
          impurity=True,
          rounded=True,
          fontsize=12) 
plt.show()

**Comparison to Euro Code**

In [ ]:
#plt.scatter(x_axis, y_predict, label='prediction')
#plt.scatter(x_axis, y_test, label='actual')
plt.scatter(y_pred_test_RF2, y_test, color='b', label='Random Forest regressor')
plt.scatter(df_EC['V_Rd'], df_EC['V_test'], color='g', label = 'EC')
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'b--', label='V_test = V_predicted')
plt.plot([0,0.8], [0,1], 'g--', label='SF = 1.22')
plt.legend()
plt.show()

In [ ]:
data = {'V_test':y_test,'V_predicted':y_pred_test_RF2}
data2 = {'V_test':df_EC['V_test'],'V_exp':df_EC['V_Rd']}

df_values = pd.DataFrame(data)
df_values2 = pd.DataFrame(data2)
print(df_values2)


In [ ]:
df_values.to_csv('../data/data1.csv', index = False)

In [ ]:
df_values2.to_csv('../data/data2.csv', index = False)